# Commodity Dashboard Notebook

Este notebook replica los tres gráficos del proyecto usando el mismo dataset generado en `public/data/commodities.json` y la misma configuración base de ECharts. Ejecuta la celda siguiente para renderizar el dashboard dentro del notebook.

In [ ]:
from pathlib import Path
import json
import uuid
from IPython.display import HTML, display

payload = json.loads(Path('public/data/commodities.json').read_text())
root_id = f'commodity-dashboard-{uuid.uuid4().hex}'
payload_json = json.dumps(payload)

html = f'''
<div id="{root_id}"></div>
<script>
(function() {{
  const payload = {payload_json};
  const root = document.getElementById('{root_id}');
  const accent = '#ffd166';
  const existing = root.querySelector('script[data-echarts-loader]');

  function formatPct(value) {{
    if (value == null || Number.isNaN(value)) return 'N/A';
    return `${{value > 0 ? '+' : ''}}${{value.toFixed(2)}}%`;
  }}

  function formatDate(value) {{
    const date = new Date(`${{value}}T00:00:00`);
    return date.toLocaleDateString('en-US', {{ year: 'numeric', month: 'short', day: 'numeric' }});
  }}

  function formatHoverDate(value) {{
    const date = new Date(`${{value}}T00:00:00`);
    return date.toLocaleDateString('en-US', {{ month: 'long', day: 'numeric' }});
  }}

  function formatPrice(value) {{
    return Number(value).toLocaleString('en-US', {{ minimumFractionDigits: 2, maximumFractionDigits: 2 }});
  }}

  function escapeHtml(value) {{
    return String(value)
      .replaceAll('&', '&amp;')
      .replaceAll('<', '&lt;')
      .replaceAll('>', '&gt;')
      .replaceAll('"', '&quot;')
      .replaceAll("'", '&#39;');
  }}

  function buildNormalizedCsv(dataset) {{
    const rows = ['year,day,date,price,change_pct'];
    dataset.series.forEach((entry) => {{
      entry.points.forEach((point) => {{
        rows.push([entry.year, point.day, point.date, point.price, point.changePct].join(','));
      }});
    }});
    return rows.join('\n');
  }}

  function ensureEcharts(callback) {{
    if (window.echarts) {{
      callback();
      return;
    }}

    const script = document.createElement('script');
    script.src = 'https://cdn.jsdelivr.net/npm/echarts@6/dist/echarts.min.js';
    script.async = true;
    script.dataset.echartsLoader = 'true';
    script.onload = callback;
    root.appendChild(script);
  }}

  function render() {{
    root.innerHTML = `
      <style>
        #${{root.id}} {{
          color-scheme: dark;
          font-family: Arial, Helvetica, sans-serif;
          color: #f2f2f2;
          background: #000;
          padding: 10px 0 20px;
        }}
        #${{root.id}} * {{ box-sizing: border-box; }}
        #${{root.id}} a {{ color: inherit; }}
        #${{root.id}} button {{ font: inherit; }}
        #${{root.id}} .app-shell {{ width: min(1000px, calc(100vw - 56px)); margin: 0 auto; display: grid; gap: 10px; }}
        #${{root.id}} .panel {{ background: #050505; border: 1px solid rgba(255,255,255,0.08); border-radius: 6px; }}
        #${{root.id}} .methodology-card {{ padding: 10px 12px; border-color: rgba(255,196,64,0.18); background: linear-gradient(180deg, rgba(12,12,12,0.98), rgba(8,8,8,0.98)); }}
        #${{root.id}} .methodology-chip {{ display: inline; color: #ffd166; font-family: 'SFMono-Regular', ui-monospace, monospace; font-size: 13px; font-weight: 700; }}
        #${{root.id}} .methodology-copy {{ display: inline; color: rgba(229,231,235,0.76); font-size: 12px; line-height: 1.4; text-align: justify; }}
        #${{root.id}} .formula-inline {{ display: block; width: 100%; margin-top: 6px; text-align: center; color: #ffe08a; font-family: 'Times New Roman', serif; font-size: 14px; }}
        #${{root.id}} .chart-grid {{ display: grid; grid-template-columns: 1fr; gap: 12px; }}
        #${{root.id}} .section-card {{ padding: 8px 10px 6px; max-width: 1000px; }}
        #${{root.id}} .section-header {{ display: flex; justify-content: space-between; gap: 10px; align-items: center; margin-bottom: 4px; padding: 2px 2px 0; }}
        #${{root.id}} .section-heading {{ min-width: 150px; }}
        #${{root.id}} .section-kicker {{ color: rgba(255,176,0,0.9); text-transform: uppercase; letter-spacing: 0.14em; font-size: 9px; margin-bottom: 3px; font-weight: 700; }}
        #${{root.id}} h2, #${{root.id}} p {{ margin: 0; }}
        #${{root.id}} .section-header h2 {{ font-size: 16px; line-height: 1; letter-spacing: -0.02em; margin-bottom: 2px; font-weight: 700; }}
        #${{root.id}} .section-header p {{ color: rgba(220,220,220,0.62); line-height: 1.3; font-size: 12px; max-width: none; }}
        #${{root.id}} .section-statline {{ display: flex; gap: 14px; align-items: center; flex-wrap: wrap; color: rgba(220,220,220,0.82); font-size: 10px; line-height: 1.2; }}
        #${{root.id}} .section-statline strong {{ color: #fff; font-size: 12px; margin-left: 4px; }}
        #${{root.id}} .section-actions {{ display: flex; gap: 6px; flex-wrap: wrap; justify-content: flex-end; }}
        #${{root.id}} .button {{ appearance: none; border: 1px solid rgba(255,255,255,0.16); background: transparent; color: #dcdcdc; text-decoration: none; padding: 6px 8px; border-radius: 4px; cursor: pointer; font-size: 10px; text-transform: uppercase; letter-spacing: 0.06em; }}
        #${{root.id}} .button:hover {{ background: rgba(255,255,255,0.06); border-color: rgba(255,255,255,0.28); }}
        #${{root.id}} .chart-canvas {{ width: 100%; height: 390px; }}
        #${{root.id}} .section-footer {{ display: flex; justify-content: space-between; gap: 12px; margin-top: 4px; color: rgba(220,220,220,0.56); font-size: 10px; }}
        @media (max-width: 1024px) {{ #${{root.id}} .section-header {{ align-items: flex-start; flex-direction: column; }} }}
        @media (max-width: 720px) {{
          #${{root.id}} .app-shell {{ width: min(100vw, calc(100vw - 18px)); }}
          #${{root.id}} .section-header, #${{root.id}} .section-footer {{ flex-direction: column; }}
          #${{root.id}} .section-actions {{ justify-content: flex-start; }}
        }}
      </style>
      <main class="app-shell">
        <section class="panel methodology-card">
          <span class="methodology-chip">Methodology:</span>
          <span class="methodology-copy">
            each line represents one calendar year. Here, “normalized” means that the first valid trading observation of the year is used as the common base level, so every subsequent point is shown as the cumulative percentage change relative to that starting value rather than as an absolute price. The x-axis is trading-day count, not calendar date, so the observation on day 40 of one year is directly comparable with day 40 of any other year. The colored line is the current year, while the grey lines represent the empirical distribution of prior-year paths, allowing current performance to be evaluated against the historical range of rallies and drawdowns observed at the same stage of the annual cycle.
          </span>
          <div class="formula-inline">YTD(y,d) = 100 · (P(y,d) / P(y,1) - 1)</div>
        </section>
        <section class="chart-grid"></section>
      </main>
    `;

    const chartGrid = root.querySelector('.chart-grid');
    const charts = [];

    ['oil', 'gold', 'sp500'].forEach((key) => {{
      const dataset = payload[key];
      const summary = dataset.summary;
      const section = document.createElement('section');
      section.className = 'panel section-card';
      const chartId = `${{root.id}}-${{key}}`;
      section.innerHTML = `
        <div class="section-header">
          <div class="section-heading">
            <div class="section-kicker">${{escapeHtml(dataset.name)}}</div>
            <h2>${{escapeHtml(dataset.name)}}</h2>
            <p>${{escapeHtml(dataset.unit)}}</p>
          </div>
          <div class="section-statline">
            <span>${{summary.currentYear}} YTD <strong>${{formatPct(summary.currentChangePct)}}</strong></span>
            <span>As of ${{formatDate(summary.currentDate)}}</span>
            <span>High ${{formatPct(summary.recordHigh?.changePct)}}</span>
            <span>Low ${{formatPct(summary.recordLow?.changePct)}}</span>
          </div>
          <div class="section-actions">
            <button class="button" data-download="${{key}}">Normalized CSV</button>
          </div>
        </div>
        <div class="chart-canvas" id="${{chartId}}"></div>
        <div class="section-footer">
          <span>Source: <a href="${{escapeHtml(dataset.sourceUrl)}}" target="_blank" rel="noreferrer">${{escapeHtml(dataset.sourceName)}}</a></span>
          <span>Coverage: ${{summary.startYear}} to ${{summary.endYear}}</span>
        </div>
      `;
      chartGrid.appendChild(section);

      section.querySelector('[data-download]').addEventListener('click', () => {{
        const blob = new Blob([buildNormalizedCsv(dataset)], {{ type: 'text/csv;charset=utf-8' }});
        const url = URL.createObjectURL(blob);
        const anchor = document.createElement('a');
        anchor.href = url;
        anchor.download = `${{dataset.key}}-normalized-ytd.csv`;
        anchor.click();
        URL.revokeObjectURL(url);
      }});

      const chart = window.echarts.init(section.querySelector(`#${{chartId}}`), undefined, {{ renderer: 'canvas' }});
      const currentYear = summary.currentYear;
      let lastMouseY = 0;
      chart.setOption({{
        animation: false,
        backgroundColor: 'transparent',
        grid: {{ left: 50, right: 14, top: 28, bottom: 30 }},
        legend: {{
          top: 14,
          left: 8,
          itemWidth: 16,
          itemHeight: 2,
          textStyle: {{ color: accent, fontFamily: 'Arial, Helvetica, sans-serif', fontSize: 10, fontWeight: 700 }},
          data: [String(currentYear)],
        }},
        tooltip: {{
          trigger: 'axis',
          axisPointer: {{
            type: 'line',
            snap: false,
            lineStyle: {{ color: 'rgba(255,255,255,0.16)', width: 1 }},
          }},
          backgroundColor: 'rgba(10, 10, 12, 0.98)',
          borderColor: 'rgba(255,255,255,0.18)',
          textStyle: {{ color: '#f4f4f4', fontFamily: 'Arial, Helvetica, sans-serif', fontSize: 10 }},
          formatter: (params) => {{
            if (!params.length) return '';
            let best = params[0];
            let bestDistance = Number.POSITIVE_INFINITY;
            for (const item of params) {{
              const [day, change] = item.data;
              const [, py] = chart.convertToPixel({{ xAxisIndex: 0, yAxisIndex: 0 }}, [day, change]);
              const distance = Math.abs(py - lastMouseY);
              if (distance < bestDistance) {{
                best = item;
                bestDistance = distance;
              }}
            }}
            const [, , date, price] = best.data;
            return [`<strong>${{best.seriesName}}</strong>`, `${{formatHoverDate(date)}}`, `Value ${{formatPrice(price)}}`].join('<br/>');
          }},
        }},
        toolbox: {{
          top: -2,
          right: 2,
          itemSize: 12,
          iconStyle: {{ borderColor: 'rgba(220,220,220,0.7)' }},
          feature: {{
            saveAsImage: {{ name: `${{dataset.key}}-ytd-by-year`, pixelRatio: 2, backgroundColor: '#070707' }},
          }},
        }},
        xAxis: {{
          type: 'value',
          name: 'Trading Days',
          nameLocation: 'middle',
          nameGap: 24,
          min: 1,
          max: Math.max(...dataset.series.map((item) => item.points.length)),
          axisLine: {{ lineStyle: {{ color: 'rgba(255,255,255,0.42)' }} }},
          axisLabel: {{ color: 'rgba(220,220,220,0.7)', fontFamily: 'Arial, Helvetica, sans-serif', fontSize: 10 }},
          splitLine: {{ show: false }},
        }},
        yAxis: {{
          type: 'value',
          name: 'YTD Change (%)',
          nameLocation: 'end',
          nameRotate: 0,
          nameGap: 38,
          nameTextStyle: {{ color: 'rgba(220,220,220,0.68)', fontFamily: 'Arial, Helvetica, sans-serif', fontSize: 10 }},
          axisLine: {{ lineStyle: {{ color: 'rgba(255,255,255,0.42)' }} }},
          axisLabel: {{ color: 'rgba(220,220,220,0.7)', fontFamily: 'Arial, Helvetica, sans-serif', fontSize: 10, formatter: (value) => `${{value}}%` }},
          splitLine: {{ lineStyle: {{ color: 'rgba(255,255,255,0.09)' }} }},
        }},
        series: dataset.series.map((entry) => {{
          const isCurrent = entry.year === currentYear;
          return {{
            name: String(entry.year),
            type: 'line',
            showSymbol: false,
            smooth: false,
            z: isCurrent ? 10 : 2,
            lineStyle: {{ color: isCurrent ? accent : 'rgba(255,255,255,0.22)', width: isCurrent ? 3.25 : 1.1 }},
            emphasis: {{ focus: 'series', lineStyle: {{ width: isCurrent ? 4 : 2 }} }},
            data: entry.points.map((point) => [point.day, point.changePct, point.date, point.price]),
          }};
        }}),
      }});

      chart.getZr().on('mousemove', (event) => {{
        lastMouseY = event.offsetY;
      }});

      charts.push(chart);
    }});

    const resize = () => charts.forEach((chart) => chart.resize());
    window.addEventListener('resize', resize, {{ passive: true }});
  }}

  ensureEcharts(render);
}})();
</script>
'''

display(HTML(html))
